# NovaCart — Silver Sellers Transformation

## 1. Import Libraries

In [0]:
from pyspark.sql import functions as F

## 2. Define Storage Paths

In [0]:
BRONZE_SELLERS_PATH = (
    "abfss://bronze@stnovacartdev.dfs.core.windows.net/"
    "olist/sellers"
)

SILVER_SELLERS_PATH = (
    "abfss://silver@stnovacartdev.dfs.core.windows.net/"
    "olist/sellers"
)

QUARANTINE_SELLERS_PATH = (
    "abfss://quarantine@stnovacartdev.dfs.core.windows.net/"
    "olist/sellers"
)

print(f"Bronze path: {BRONZE_SELLERS_PATH}")
print(f"Silver path: {SILVER_SELLERS_PATH}")
print(f"Quarantine path: {QUARANTINE_SELLERS_PATH}")

## 3. Read Bronze Sellers Data

In [0]:
sellers_bronze_df = (
    spark.read
    .format("delta")
    .load(BRONZE_SELLERS_PATH)
)

bronze_row_count = sellers_bronze_df.count()

print("Bronze sellers loaded successfully.")
print(f"Bronze row count: {bronze_row_count}")

sellers_bronze_df.printSchema()
display(sellers_bronze_df.limit(10))

## 4. Validate Required Columns

In [0]:
required_columns = [
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "_source_file",
    "_ingestion_timestamp",
    "_batch_id",
]

missing_columns = [
    column
    for column in required_columns
    if column not in sellers_bronze_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required Bronze columns: {missing_columns}"
    )

print("Required-column validation passed.")

## 5. Profile Missing and Invalid Seller Values

In [0]:
sellers_profile_df = sellers_bronze_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        (
            F.col("seller_id").isNull()
            | (F.trim(F.col("seller_id")) == "")
        ).cast("int")
    ).alias("invalid_seller_id"),

    F.sum(
        (
            F.col("seller_zip_code_prefix").isNull()
            | (F.col("seller_zip_code_prefix") < 0)
            | (F.col("seller_zip_code_prefix") > 99999)
        ).cast("int")
    ).alias("invalid_seller_zip_code_prefix"),

    F.sum(
        (
            F.col("seller_city").isNull()
            | (F.trim(F.col("seller_city")) == "")
        ).cast("int")
    ).alias("invalid_seller_city"),

    F.sum(
        (
            F.col("seller_state").isNull()
            | (F.trim(F.col("seller_state")) == "")
        ).cast("int")
    ).alias("invalid_seller_state"),

    F.sum(
        (
            F.col("seller_state").isNotNull()
            & ~F.trim(F.col("seller_state")).rlike("^[A-Za-z]{2}$")
        ).cast("int")
    ).alias("invalid_state_format"),
)

display(sellers_profile_df)

## 6. Print Full Seller Quality Profile

In [0]:
profile = sellers_profile_df.first().asDict()

for metric, value in profile.items():
    print(f"{metric}: {value}")

## 7. Check Duplicate Seller IDs

In [0]:
duplicate_seller_ids_df = (
    sellers_bronze_df
    .groupBy("seller_id")
    .count()
    .filter(
        F.col("seller_id").isNotNull()
        & (F.col("count") > 1)
    )
)

duplicate_seller_id_count = duplicate_seller_ids_df.count()

print(
    f"Number of seller_id values appearing more than once: "
    f"{duplicate_seller_id_count}"
)

display(duplicate_seller_ids_df.limit(20))

## 8. Check Exact Duplicate Records

In [0]:
business_columns = [
    "seller_id",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
]

exact_duplicate_count = (
    bronze_row_count
    - sellers_bronze_df
        .dropDuplicates(business_columns)
        .count()
)

print(f"Exact duplicate seller rows: {exact_duplicate_count}")

## 9. Clean and Standardize Seller Fields

In [0]:
sellers_cleaned_df = (
    sellers_bronze_df
    .withColumn(
        "seller_id",
        F.trim(F.col("seller_id"))
    )
    .withColumn(
        "seller_city",
        F.lower(F.trim(F.col("seller_city")))
    )
    .withColumn(
        "seller_state",
        F.upper(F.trim(F.col("seller_state")))
    )
)

## 10. Define Seller Validation Rules

In [0]:
invalid_seller_id_condition = (
    F.col("seller_id").isNull()
    | (F.col("seller_id") == "")
)

invalid_seller_zip_code_condition = (
    F.col("seller_zip_code_prefix").isNull()
    | (F.col("seller_zip_code_prefix") < 0)
    | (F.col("seller_zip_code_prefix") > 99999)
)

invalid_seller_city_condition = (
    F.col("seller_city").isNull()
    | (F.col("seller_city") == "")
)

invalid_seller_state_condition = (
    F.col("seller_state").isNull()
    | (F.col("seller_state") == "")
    | ~F.col("seller_state").rlike("^[A-Z]{2}$")
)

## 11. Assign Seller Rejection Reasons

In [0]:
sellers_validated_df = sellers_cleaned_df.withColumn(
    "_rejection_reason",

    F.when(
        invalid_seller_id_condition,
        F.lit("MISSING_SELLER_ID")
    )
    .when(
        invalid_seller_zip_code_condition,
        F.lit("INVALID_SELLER_ZIP_CODE_PREFIX")
    )
    .when(
        invalid_seller_city_condition,
        F.lit("MISSING_SELLER_CITY")
    )
    .when(
        invalid_seller_state_condition,
        F.lit("INVALID_SELLER_STATE")
    )
    .otherwise(F.lit(None))
)

## 12. Review Seller Validation Results

In [0]:
display(
    sellers_validated_df
    .groupBy("_rejection_reason")
    .count()
    .orderBy("_rejection_reason")
)

## 13. Split Valid and Invalid Sellers

In [0]:
sellers_valid_df = (
    sellers_validated_df
    .filter(F.col("_rejection_reason").isNull())
    .drop("_rejection_reason")
)

sellers_quarantine_df = (
    sellers_validated_df
    .filter(F.col("_rejection_reason").isNotNull())
)

## 14. Add Silver Processing Metadata

In [0]:
sellers_silver_df = (
    sellers_valid_df
    .withColumn(
        "_silver_processed_at",
        F.current_timestamp()
    )
)

## 15. Add Quarantine Metadata

In [0]:
sellers_quarantine_df = (
    sellers_quarantine_df
    .withColumn(
        "_quarantined_at",
        F.current_timestamp()
    )
    .withColumn(
        "_source_dataset",
        F.lit("sellers")
    )
)

## 16. Count Silver and Quarantine Records

In [0]:
valid_row_count = sellers_silver_df.count()
quarantine_row_count = sellers_quarantine_df.count()

print(f"Valid Silver rows: {valid_row_count}")
print(f"Quarantined rows: {quarantine_row_count}")
print(f"Bronze input rows: {bronze_row_count}")

## 17. Validate Row-Count Reconciliation

In [0]:
if valid_row_count + quarantine_row_count != bronze_row_count:
    raise ValueError(
        "Row-count validation failed: "
        "Silver rows + quarantine rows do not equal Bronze input rows."
    )

print("Row-count validation passed.")

## 18. Write Valid Sellers to Silver

In [0]:
(
    sellers_silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(SILVER_SELLERS_PATH)
)

print("Silver sellers written successfully.")

## 19. Write Invalid Sellers to Quarantine

In [0]:
(
    sellers_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(QUARANTINE_SELLERS_PATH)
)

print("Sellers quarantine output written successfully.")

## 20. Read Written Delta Outputs

In [0]:
sellers_silver_written_df = (
    spark.read
    .format("delta")
    .load(SILVER_SELLERS_PATH)
)

sellers_quarantine_written_df = (
    spark.read
    .format("delta")
    .load(QUARANTINE_SELLERS_PATH)
)

silver_written_count = sellers_silver_written_df.count()
quarantine_written_count = sellers_quarantine_written_df.count()

print(f"Written Silver rows: {silver_written_count}")
print(f"Written quarantine rows: {quarantine_written_count}")

## 21. Validate Written Outputs

In [0]:
if silver_written_count != valid_row_count:
    raise ValueError(
        "Silver write validation failed: "
        f"expected {valid_row_count}, wrote {silver_written_count}."
    )

if quarantine_written_count != quarantine_row_count:
    raise ValueError(
        "Quarantine write validation failed: "
        f"expected {quarantine_row_count}, wrote "
        f"{quarantine_written_count}."
    )

if silver_written_count + quarantine_written_count != bronze_row_count:
    raise ValueError(
        "Final reconciliation failed: "
        "Silver + quarantine does not equal Bronze."
    )

print("Silver sellers pipeline completed successfully.")
print("Final row-count validation passed.")

## 22. Inspect Final Silver Sellers Dataset

In [0]:
sellers_silver_written_df.printSchema()

display(
    sellers_silver_written_df.select(
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
        "_silver_processed_at"
    ).limit(20)
)